# Pipeline combinado (provisional): U-Net + limpieza morfológica + HSV/Lab

Notebook de trabajo para ir armando, bloque por bloque, un pipeline que combine:
1. Predicción de agua con la U-Net entrenada (con su IoU)
2. Limpieza morfológica de la máscara (cerrar huecos, suavizar bordes)
3. Refinamiento con el segmentador clásico HSV/Lab de `water_masc1`

**Por ahora** las imágenes se buscan por nombre dentro de `river_water_index.csv` (así podemos calcular IoU contra su máscara real). Más adelante se adapta para cargar cualquier imagen local, tenga o no máscara de referencia.

## Configuración

In [ ]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

BASE_PATH = Path(r"D:\proyecto_eutrofizacion")
sys.path.insert(0, str(BASE_PATH / "src"))

from unet.config import UNetConfig
from unet.preprocessing import load_image_rgb, load_mask_binary, downscale_image, downscale_mask
from unet.patches import get_patch_positions
from unet.reconstruction import reconstruct_full_prediction
from unet.augmentation import get_val_transforms
from unet.metrics import compute_metrics
from unet.inference import load_model

from water_masc1 import segmentar_agua
from water_masc2 import conectar_graffiti_y_cerrar, suavizar_bordes_contorno

# NOTA: no importamos unet.visualization aquí a propósito -- ese módulo fuerza
# el backend "Agg" de matplotlib (pensado para guardar figuras sin pantalla
# durante el entrenamiento), lo que rompería la visualización inline en este
# notebook. Usamos nuestro propio helper de plotting más abajo.

config = UNetConfig(base_path=BASE_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df_river = pd.read_csv(config.get_path(config.csv_path))
print(f"Índice cargado: {len(df_river)} imágenes con máscara disponible")


In [ ]:
def buscar_imagen_por_nombre(nombre, df=df_river):
    """Busca una imagen por nombre de archivo en el índice del dataset.
    Devuelve la fila del CSV (con filepath y segmentation_mask_path)."""
    filas = df[df["filepath"].str.contains(nombre, regex=False)]
    if filas.empty:
        raise ValueError(f"No se encontró la imagen '{nombre}' en el índice.")
    return filas.iloc[0]


def mostrar_paneles(image_ds, paneles, suptitle=None):
    """Muestra la imagen original + N máscaras/paneles lado a lado.
    paneles: lista de tuplas (titulo, array, cmap).
    """
    n = len(paneles) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    axes[0].imshow(image_ds)
    axes[0].set_title("Imagen")
    axes[0].axis("off")

    for ax, (titulo, arr, cmap) in zip(axes[1:], paneles):
        ax.imshow(arr, cmap=cmap)
        ax.set_title(titulo)
        ax.axis("off")

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


# Diccionario donde se van acumulando los resultados de cada imagen a medida
# que avanzamos por los bloques (cada bloque le añade sus propias claves).
resultados = {}

print("Listo: buscar_imagen_por_nombre(), mostrar_paneles() y 'resultados' definidos.")


## Predicción con U-Net (IoU)

Recibe una lista de nombres de imagen (por ahora, deben existir en `river_water_index.csv` para poder comparar contra su máscara real). Predice la máscara de agua con la U-Net y calcula el IoU de cada una.

In [ ]:
# === IMÁGENES A PROCESAR ===
# Cambia estos nombres por los que quieras probar (deben existir en el índice
# del dataset). Cuando adaptemos el pipeline para cargar imágenes sueltas del
# local, esta lista dejará de depender de river_water_index.csv.
IMAGENES = [
    "DJI_10226.JPG",
    "DJI_11677.JPG",
    "DJI_2305.JPG",
    "DJI_10674.JPG",
    "DJI_9805.JPG",
]

# --- Modelo ---
checkpoint_path = config.get_path(config.checkpoints_dir) / "best_model.pth"
transform = get_val_transforms(config.imagenet_mean, config.imagenet_std)

model = load_model(
    checkpoint_path,
    encoder_name=config.encoder_name,
    encoder_weights=None,
    device=device,
)

# --- Predicción por imagen ---
for nombre in IMAGENES:
    fila = buscar_imagen_por_nombre(nombre)
    img_path = BASE_PATH / fila["filepath"]
    mask_path = BASE_PATH / fila["segmentation_mask_path"]

    image = load_image_rgb(img_path)
    h, w = image.shape[:2]
    mask_gt = load_mask_binary(mask_path, h, w, BASE_PATH)

    image_ds = downscale_image(image, config.downscale_factor)
    mask_gt_ds = downscale_mask(mask_gt, config.downscale_factor)
    h_ds, w_ds = image_ds.shape[:2]

    positions = get_patch_positions(h_ds, w_ds, config.patch_size, config.stride)
    prob_avg, mask_unet = reconstruct_full_prediction(
        model=model,
        image_ds=image_ds,
        positions=positions,
        transform=transform,
        patch_size=config.patch_size,
        threshold=config.threshold,
        batch_size=config.batch_size * 2,
        device=device,
    )

    gt_float = (mask_gt_ds > 127).astype(np.float32)
    m = compute_metrics(prob_avg, gt_float, threshold=config.threshold)

    resultados[nombre] = {
        "image_ds": image_ds,
        "mask_gt_ds": mask_gt_ds,
        "mask_unet": mask_unet,
        "iou_unet": m.iou,
        "dice_unet": m.dice,
    }

    print(f"{nombre}: {m}")

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("Ground truth", r["mask_gt_ds"], "gray"),
            ("Predicción U-Net", r["mask_unet"], "gray"),
        ],
        suptitle=f"{nombre}  —  IoU={r['iou_unet']:.4f}  |  Dice={r['dice_unet']:.4f}",
    )


## Cerrar máscaras

Aplica `conectar_graffiti_y_cerrar` (cierra huecos/gaps pequeños) y `suavizar_bordes_contorno` (suaviza el contorno) sobre la máscara que predijo la U-Net.

**Ojo**: `suavizar_bordes_contorno` se queda solo con el contorno EXTERNO más grande — si la máscara tiene varias zonas de agua desconectadas entre sí, las más pequeñas se pierden. Lo dejamos así por ahora porque es la función que ya tenías; lo ajustamos si al ver los resultados no conviene.

In [ ]:
for nombre, r in resultados.items():
    mask_cerrada = conectar_graffiti_y_cerrar(
        r["mask_unet"], skeleton_kernel=3, dilation_kernel=30
    )
    mask_cerrada = suavizar_bordes_contorno(mask_cerrada, contour_approx=7)

    gt_float = (r["mask_gt_ds"] > 127).astype(np.float32)
    m = compute_metrics(mask_cerrada, gt_float, threshold=0.5)

    r["mask_cerrada"] = mask_cerrada
    r["iou_cerrada"] = m.iou
    r["dice_cerrada"] = m.dice

    print(f"{nombre}: IoU antes={r['iou_unet']:.4f}  ->  después de cerrar={m.iou:.4f}")

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("U-Net (antes)", r["mask_unet"], "gray"),
            ("Cerrada (después)", r["mask_cerrada"], "gray"),
        ],
        suptitle=(
            f"{nombre}  —  IoU antes={r['iou_unet']:.4f}  "
            f"->  después={r['iou_cerrada']:.4f}"
        ),
    )


## Parámetros HSV y demás

Genera una segunda máscara con el segmentador clásico Lab+HSV de `water_masc1.segmentar_agua` (los mismos parámetros que ya usabas en `01_water_segmentation.ipynb`), y la combina con la máscara ya cerrada del bloque anterior.

La combinación es una **unión (OR)**: agrega como agua cualquier píxel que cualquiera de los dos métodos detecte — pensado para recuperar agua que la U-Net pudo haber perdido. Si en cambio prefieres ser más conservador (solo contar agua donde ambos coinciden), cambia `cv2.bitwise_or` por `cv2.bitwise_and` más abajo.

In [ ]:
# Mismos parámetros HSV/Lab que en 01_water_segmentation.ipynb / 02_superpixels_training.ipynb
PARAMETROS_HSV = {
    "a_min": -120,
    "a_max": 1,
    "b_min": -120,
    "b_max": 14,
    "l_min": 50,
    "l_max": 255,
    "h_min": 1,
    "h_max": 180,
    "s_min": 0,
    "s_max": 110,
}

for nombre, r in resultados.items():
    fila = buscar_imagen_por_nombre(nombre)
    ruta_imagen = BASE_PATH / fila["filepath"]

    # segmentar_agua espera BGR (igual que cv2.imread lo carga de forma nativa)
    imagen_bgr = cv2.imread(str(ruta_imagen))

    # segmentar_agua devuelve la máscara en la resolución ORIGINAL de la imagen
    # (downscalea y vuelve a subir internamente); la bajamos al mismo tamaño
    # (x2) que usa la U-Net para poder combinarlas píxel a píxel.
    mask_hsv_orig = segmentar_agua(imagen_bgr, **PARAMETROS_HSV, downscale=4, verbose=False)
    h_ds, w_ds = r["mask_cerrada"].shape
    mask_hsv = cv2.resize(mask_hsv_orig, (w_ds, h_ds), interpolation=cv2.INTER_NEAREST)

    mask_combinada = cv2.bitwise_or(r["mask_cerrada"], mask_hsv)

    gt_float = (r["mask_gt_ds"] > 127).astype(np.float32)
    m_hsv = compute_metrics(mask_hsv, gt_float, threshold=0.5)
    m_comb = compute_metrics(mask_combinada, gt_float, threshold=0.5)

    r["mask_hsv"] = mask_hsv
    r["mask_combinada"] = mask_combinada
    r["iou_hsv"] = m_hsv.iou
    r["iou_combinada"] = m_comb.iou

    print(
        f"{nombre}: IoU cerrada={r['iou_cerrada']:.4f}  |  "
        f"HSV={m_hsv.iou:.4f}  |  combinada={m_comb.iou:.4f}"
    )

# --- Visualización ---
for nombre, r in resultados.items():
    mostrar_paneles(
        r["image_ds"],
        [
            ("Cerrada (U-Net)", r["mask_cerrada"], "gray"),
            ("HSV/Lab", r["mask_hsv"], "gray"),
            ("Combinada (OR)", r["mask_combinada"], "gray"),
        ],
        suptitle=(
            f"{nombre}  —  cerrada={r['iou_cerrada']:.4f}  |  "
            f"HSV={r['iou_hsv']:.4f}  |  combinada={r['iou_combinada']:.4f}"
        ),
    )
